In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.io import fits
import glob
from sft import *
from datetime import datetime, timedelta
from interpolation import interp1d
from utils import *
from scipy.ndimage import gaussian_filter

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
def show(data):
    x_lims = mdates.date2num([datetime(2024,1,1), datetime(2026,1,1)])
    y_lims = [-1, 1]

    fig, ax = plt.subplots(figsize=(14,9))
    ax.imshow(data, cmap='seismic', vmin=-10, vmax=10, #interpolation='bicubic',
              extent=[x_lims[0], x_lims[1], y_lims[0], y_lims[1]], aspect='auto', origin='lower')

    ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y/%m'))

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

In [3]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*')))

flux = []
dates = []

for file in files:

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    data = rebin(data, 2)
    data = np.nanmean(data, axis=-1)
    data = (data[1:] + data[:-1]) / 2
    data = np.nan_to_num(data)

    date = datetime.fromisoformat(header['T_OBS'][:-4].replace('.', '').replace('_', 'T'))

    flux.append(data)
    dates.append(date)

flux = np.array(flux)
dates = np.array(dates)

/tmp/ipykernel_6838/889444508.py:13: RuntimeWarning: Mean of empty slice
  data = np.nanmean(data, axis=-1)


In [4]:
dsine = 1 / 359.5
lati = np.arange(-1,1 + dsine / 2, dsine)
lati = np.arcsin(lati.clip(-1,1)) * 180 / np.pi
lat = (lati[1:] + lati[:-1]) / 2

In [5]:
show(flux.T)

In [6]:
R = 695e8
u = 1000
D = 500e10
lam = 45
dt = timedelta(days=1).total_seconds()

xi = lati * np.pi / 180 * R
vi = u * flow(lati, a=0.5)
li = 2 * np.pi * R * np.cos(lati  * np.pi / 180)

y = flux[0].copy()
Q = []
T = []

for t in np.arange(dates[0], dates[-1], timedelta(days=1)):
    y_ = interp1d(flux, dates, t)
    y = advect(y, vi, dt, xi=xi, ai=li)
    y = diffuse(y, D, dt, xi=xi, ai=li)
    y[np.abs(lat) < lam] = y_[np.abs(lat) < lam]

    Q.append(y)
    T.append(t)

Q = np.array(Q)
T = np.array(T)

In [7]:
show(Q.T)

In [9]:
i = -1

plt.figure(figsize=(10,8))
plt.plot(lat, flux[0])
plt.plot(lat, flux[i])
plt.plot(lat, interp1d(Q, T, dates[i]))
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [12]:
Q.shape

(737, 719)

In [1]:
from scipy.linalg import solve_banded

?solve_banded